# Ablation Studies

**Sections:**
1. MoE ablation — GPS++ with/without Mixture of Experts at same scale
2. Pairformer size ablation — scaling behavior across model sizes
3. Pairformer vs Pairmixer — speed and quality comparison

---
## 1. MoE Ablation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

from notebook_utils import (
    ADMET_TASK_ORDER, TASK_METRICS, TASK_CATEGORY, TDC_SOTA,
    load_results, get_sota, infer_model_name, infer_model_family,
    MODEL_REGISTRY,
)

df = pd.read_csv('../results/experiment_results.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
print(f'Loaded {len(df)} rows')

In [ ]:
# Task definitions
TASKS = {
    'caco2_wang':      ('graph_caco2_wang/mae/test', 'lower', 'Absorption', 'MAE'),
    'bbb_martins':     ('graph_bbb_martins/auroc/test', 'higher', 'Distribution', 'AUROC'),
    'cyp3a4_veith':    ('graph_cyp3a4_veith/auprc/test', 'higher', 'Metabolism', 'AUPRC'),
    'half_life_obach': ('graph_half_life_obach/spearman/test', 'higher', 'Excretion', 'Spearman'),
    'herg':            ('graph_herg/auroc/test', 'higher', 'Toxicity', 'AUROC'),
}

TDC_SOTA = {
    'caco2_wang':      (0.256, 0.006),
    'bbb_martins':     (0.924, 0.003),
    'cyp3a4_veith':    (0.916, 0.000),
    'half_life_obach': (0.576, 0.025),
    'herg':            (0.880, 0.002),
}

TASK_ORDER = list(TASKS.keys())

In [ ]:
# Extract results for each condition
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

def get_latest(subset, task, col):
    """Get the latest result for a task from a subset."""
    rows = subset[subset['task'] == task].sort_values('timestamp')
    vals = pd.to_numeric(rows[col], errors='coerce').dropna()
    return vals.iloc[-1] if len(vals) > 0 else np.nan

def get_best(subset, task, col, direction):
    """Get the best result for a task from a subset."""
    rows = subset[subset['task'] == task]
    vals = pd.to_numeric(rows[col], errors='coerce').dropna()
    if len(vals) == 0:
        return np.nan
    return vals.min() if direction == 'lower' else vals.max()

# Conditions
moe_ft = df[df['wandb_tags'].str.contains('moe.*finetune', case=False, na=False)]
scratch_768 = df[df['wandb_tags'].str.contains('moe_baseline.*768d8L', case=False, na=False)]
gpspp_800M_ft = df[
    (df['hidden_dim'] == 1536) & (df['gnn_depth'] == 12) &
    (df['is_finetuning'] == True)
]

print(f'MoE FT rows: {len(moe_ft)}')
print(f'Scratch 768d/8L rows: {len(scratch_768)}')
print(f'GPS++ 800M FT rows: {len(gpspp_800M_ft)}')

### Results Table

In [ ]:
# Build results table
rows = []
for task in TASK_ORDER:
    col, direction, category, metric_name = TASKS[task]
    arrow = '\u2191' if direction == 'higher' else '\u2193'
    sota_mean, sota_std = TDC_SOTA[task]

    scratch_v = get_latest(scratch_768, task, col)
    moe_v = get_latest(moe_ft, task, col)
    g800_v = get_best(gpspp_800M_ft, task, col, direction)

    rows.append({
        'Task': task,
        'Category': category,
        f'Metric ({arrow})': metric_name,
        'TDC SOTA': f'{sota_mean:.3f} \u00b1 {sota_std:.3f}',
        'Scratch (768d/8L)': scratch_v,
        'MoE FT (768d/8L)': moe_v,
        'GPS++ 800M best': g800_v,
    })

results = pd.DataFrame(rows).set_index('Task')

# Style: bold best among Scratch and MoE (the controlled comparison)
def highlight_moe_comparison(row):
    styles = [''] * len(row)
    task = row.name
    _, direction, _, _ = TASKS[task]
    s_col = 'Scratch (768d/8L)'
    m_col = 'MoE FT (768d/8L)'

    s_idx = list(results.columns).index(s_col)
    m_idx = list(results.columns).index(m_col)

    s_val = row[s_col]
    m_val = row[m_col]

    if pd.notna(s_val) and pd.notna(m_val):
        if direction == 'lower':
            winner = s_idx if s_val < m_val else m_idx
        else:
            winner = s_idx if s_val > m_val else m_idx
        styles[winner] = 'font-weight: bold; color: #1a5fb4'
    elif pd.notna(m_val):
        styles[m_idx] = 'font-weight: bold; color: #1a5fb4'
    elif pd.notna(s_val):
        styles[s_idx] = 'font-weight: bold; color: #1a5fb4'

    return styles

styled = (
    results.style
    .format(precision=4, na_rep='(pending)',
            subset=['Scratch (768d/8L)', 'MoE FT (768d/8L)', 'GPS++ 800M best'])
    .apply(highlight_moe_comparison, axis=1)
    .set_caption('MoE Ablation: Same architecture (768d/8L), with vs without MoE pre-training')
    .set_table_styles([
        {'selector': 'th', 'props': [('font-weight', 'bold')]},
        {'selector': 'td', 'props': [('text-align', 'center')]},
    ])
)

print('Blue = winner of Scratch vs MoE (controlled comparison)')
print('GPS++ 800M shown for reference (different model scale)')
display(styled)

### Per-task bar chart

In [ ]:
conditions = ['Scratch (768d/8L)', 'MoE FT (768d/8L)', 'GPS++ 800M best']
cond_colors = ['#999999', '#EB423D', '#466eff']

fig, axes = plt.subplots(1, 5, figsize=(24, 5))

for ax, task in zip(axes, TASK_ORDER):
    col, direction, category, metric_name = TASKS[task]
    arrow = '\u2191' if direction == 'higher' else '\u2193'
    sota_mean, sota_std = TDC_SOTA[task]

    vals = [results.loc[task, c] for c in conditions]

    # Find best among scratch and MoE
    controlled = [vals[0], vals[1]]
    controlled_finite = [(i, v) for i, v in enumerate(controlled) if pd.notna(v)]
    best_controlled = None
    if len(controlled_finite) >= 2:
        if direction == 'lower':
            best_controlled = min(controlled_finite, key=lambda x: x[1])[0]
        else:
            best_controlled = max(controlled_finite, key=lambda x: x[1])[0]

    for i, (v, c) in enumerate(zip(vals, cond_colors)):
        if pd.notna(v):
            ec = 'black' if i == best_controlled else c
            lw = 2.5 if i == best_controlled else 0.5
            ax.bar(i, v, color=c, edgecolor=ec, linewidth=lw, width=0.65)

    # SOTA line
    ax.axhline(y=sota_mean, color='black', linewidth=2.5, linestyle=':', zorder=5)
    ax.axhspan(sota_mean - sota_std, sota_mean + sota_std,
               alpha=0.15, color='black', zorder=1)

    # Y-axis limits: zoom to data range
    all_v = [v for v in vals if pd.notna(v)] + [sota_mean]
    vmin, vmax = min(all_v), max(all_v)
    span = vmax - vmin if vmax > vmin else vmax * 0.1
    ax.set_ylim(max(0, vmin - span * 0.15), vmax + span * 0.15)

    ax.set_xticks(range(3))
    ax.set_xticklabels(['Scratch', 'MoE FT', '800M best'], rotation=30, ha='right', fontsize=10)
    ax.set_title(f'{task}\n({category})', fontsize=12, fontweight='bold')
    ax.set_ylabel(f'{metric_name} {arrow}', fontsize=11)
    ax.tick_params(axis='y', labelsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, axis='y', ls='--', alpha=0.3)

# Legend
legend_handles = [Patch(facecolor=c, label=l) for c, l in zip(cond_colors, conditions)]
legend_handles.append(Line2D([0], [0], color='black', linewidth=2.5, linestyle=':', label='TDC SOTA'))
fig.legend(handles=legend_handles, loc='lower center', ncol=4, fontsize=11,
           bbox_to_anchor=(0.5, -0.08), frameon=True)

fig.suptitle('MoE Ablation: GPS++ 768d/8L with vs without MoE Pre-training',
             fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()

### Summary

In [ ]:
# Count wins: MoE vs Scratch (controlled comparison)
moe_wins = 0
scratch_wins = 0
ties = 0
pending = 0

print(f'{"Task":<25s} {"Scratch":>10s} {"MoE FT":>10s} {"Winner":>15s} {"Delta":>10s}')
print('=' * 75)

for task in TASK_ORDER:
    col, direction, category, metric_name = TASKS[task]
    s_v = results.loc[task, 'Scratch (768d/8L)']
    m_v = results.loc[task, 'MoE FT (768d/8L)']

    if pd.isna(s_v) or pd.isna(m_v):
        winner = '(pending)'
        delta_str = ''
        pending += 1
    else:
        if direction == 'lower':
            delta = s_v - m_v  # positive = MoE better
        else:
            delta = m_v - s_v  # positive = MoE better

        if abs(delta) < 1e-6:
            winner = 'tie'
            ties += 1
        elif delta > 0:
            winner = 'MoE'
            moe_wins += 1
        else:
            winner = 'Scratch'
            scratch_wins += 1
        delta_str = f'{delta:+.4f}'

    s_str = f'{s_v:.4f}' if pd.notna(s_v) else '(pending)'
    m_str = f'{m_v:.4f}' if pd.notna(m_v) else '(pending)'
    print(f'{task:<25s} {s_str:>10s} {m_str:>10s} {winner:>15s} {delta_str:>10s}')

print()
total = moe_wins + scratch_wins + ties
print(f'MoE wins: {moe_wins}/{total}  |  Scratch wins: {scratch_wins}/{total}  |  Ties: {ties}/{total}')
if pending > 0:
    print(f'Pending: {pending} tasks (run scripts/05_scratch_moe_baseline.sh)')

---
## 2. Pairformer Size Ablation

In [ ]:
# Re-read CSV (picks up new runs since cell 2)
df = pd.read_csv('../results/experiment_results.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
print(f'Loaded {len(df)} rows')

In [ ]:
# Task definitions (same 5 representative tasks)
TASKS = {
    'caco2_wang':        ('graph_caco2_wang/mae/test', 'lower', 'Absorption', 'MAE'),
    'pgp_broccatelli':   ('graph_pgp_broccatelli/auroc/test', 'higher', 'Absorption', 'AUROC'),
    'solubility_aqsoldb':('graph_solubility_aqsoldb/mae/test', 'lower', 'Absorption', 'MAE'),
    'half_life_obach':   ('graph_half_life_obach/spearman/test', 'higher', 'Excretion', 'Spearman'),
    'herg':              ('graph_herg/auroc/test', 'higher', 'Toxicity', 'AUROC'),
}
TASK_ORDER = list(TASKS.keys())

TDC_SOTA = {
    'caco2_wang':        (0.256, 0.006),
    'pgp_broccatelli':   (0.938, 0.002),
    'solubility_aqsoldb':(0.741, 0.013),
    'half_life_obach':   (0.576, 0.025),
    'herg':              (0.880, 0.002),
}

# Model variants ordered by size
MODEL_VARIANTS = [
    ('pairformer',        96,  16, '#64B478'),
    ('pairformer_17M',256,  12, '#FF883D'),
    ('pairformer_52M', 384,  16, '#EB423D'),
    ('pairformer_boltz', 384,  48, '#9B59B6'),
]

def classify_model(row):
    tags = str(row.get('wandb_tags', ''))
    for name, _, _, _ in MODEL_VARIANTS:
        if name in tags:
            return name
    dim, depth = int(row.get('hidden_dim', 0)), int(row.get('gnn_depth', 0))
    if dim == 96 and depth == 16:
        return 'pairformer'
    return None

In [ ]:
# Filter to pairformer rows
pf = df[df['model'] == 'pyg:pairformer'].copy()
pf['variant'] = pf.apply(classify_model, axis=1)
pf = pf[pf['variant'].notna()]
pf['is_ft'] = pf['is_finetuning'].fillna(False).astype(bool)

# Scratch runs only for the size ablation
scratch = pf[~pf['is_ft']].copy()
# Keep latest per (variant, task)
scratch = scratch.sort_values('timestamp').drop_duplicates(
    subset=['variant', 'task'], keep='last'
).reset_index(drop=True)

# FT runs for pairformer_boltz (for reference)
boltz_ft = pf[(pf['variant'] == 'pairformer_boltz') & pf['is_ft']].copy()
boltz_ft = boltz_ft.sort_values('timestamp').drop_duplicates(
    subset=['task'], keep='last'
).reset_index(drop=True)

print(f'Scratch rows: {len(scratch)}')
for v, _, _, _ in MODEL_VARIANTS:
    n = len(scratch[scratch['variant'] == v])
    print(f'  {v}: {n} tasks')
print(f'Boltz FT rows: {len(boltz_ft)}')

### 1. Results Table

In [ ]:
def get_val(subset, task, col, variant=None):
    if variant is not None:
        rows = subset[subset['variant'] == variant]
    else:
        rows = subset
    rows = rows[rows['task'] == task]
    vals = pd.to_numeric(rows[col], errors='coerce').dropna()
    return vals.iloc[-1] if len(vals) > 0 else np.nan

# Build table
table_rows = []
for task in TASK_ORDER:
    col, direction, category, metric_name = TASKS[task]
    arrow = '\u2191' if direction == 'higher' else '\u2193'
    sota_mean, sota_std = TDC_SOTA[task]

    row = {
        'Task': task,
        'Category': category,
        f'Metric': f'{metric_name} {arrow}',
        'TDC SOTA': f'{sota_mean:.3f} \u00b1 {sota_std:.3f}',
    }
    for name, dim, depth, _ in MODEL_VARIANTS:
        row[f'{name}\n({dim}d/{depth}L)'] = get_val(scratch, task, col, variant=name)

    # Boltz FT
    row['boltz FT\n(rxrx3)'] = get_val(boltz_ft, task, col)

    table_rows.append(row)

results = pd.DataFrame(table_rows).set_index('Task')

# Find best scratch variant per task
scratch_cols = [f'{n}\n({d}d/{dp}L)' for n, d, dp, _ in MODEL_VARIANTS]

def highlight_best_scratch(row):
    task = row.name
    _, direction, _, _ = TASKS[task]
    styles = [''] * len(row)
    cols = list(results.columns)

    # Find best among scratch variants
    scratch_vals = {c: row[c] for c in scratch_cols if c in cols and pd.notna(row[c])}
    if scratch_vals:
        if direction == 'lower':
            best = min(scratch_vals, key=scratch_vals.get)
        else:
            best = max(scratch_vals, key=scratch_vals.get)
        j = cols.index(best)
        styles[j] = 'font-weight: bold; color: #1a5fb4'

    return styles

def sota_style(col):
    if col.name == 'TDC SOTA':
        return ['background-color: #f0f0f0; color: #555; font-size: 0.9em'] * len(col)
    return [''] * len(col)

fmt_cols = [c for c in results.columns if c not in ('Category', 'Metric', 'TDC SOTA')]
styled = (
    results.style
    .format(precision=4, na_rep='', subset=fmt_cols)
    .apply(highlight_best_scratch, axis=1)
    .apply(sota_style, axis=0)
    .set_caption('Pairformer Size Ablation — Scratch (no pre-training)')
    .set_table_styles([
        {'selector': 'th', 'props': [('font-weight', 'bold')]},
        {'selector': 'td', 'props': [('text-align', 'center')]},
    ])
)
print('Blue = best among scratch variants')
display(styled)

### 2. Per-task bar chart

In [ ]:
fig, axes = plt.subplots(1, len(TASK_ORDER), figsize=(28, 6))

bar_labels = [f'{n}\n({d}d/{dp}L)' for n, d, dp, _ in MODEL_VARIANTS]
bar_colors = [c for _, _, _, c in MODEL_VARIANTS]
n_bars = len(MODEL_VARIANTS)

for ax, task in zip(axes, TASK_ORDER):
    col, direction, category, metric_name = TASKS[task]
    arrow = '\u2191' if direction == 'higher' else '\u2193'
    sota_mean, sota_std = TDC_SOTA[task]

    vals = []
    for name, _, _, _ in MODEL_VARIANTS:
        vals.append(get_val(scratch, task, col, variant=name))

    # Find best scratch
    finite = [(i, v) for i, v in enumerate(vals) if pd.notna(v)]
    best_idx = None
    if finite:
        if direction == 'lower':
            best_idx = min(finite, key=lambda x: x[1])[0]
        else:
            best_idx = max(finite, key=lambda x: x[1])[0]

    for i, (v, c) in enumerate(zip(vals, bar_colors)):
        if pd.notna(v):
            ec = 'black' if i == best_idx else c
            lw = 2.5 if i == best_idx else 0.5
            ax.bar(i, v, color=c, edgecolor=ec, linewidth=lw, width=0.7)

    # SOTA line
    ax.axhline(y=sota_mean, color='black', linewidth=2.5, linestyle=':', zorder=5)
    ax.axhspan(sota_mean - sota_std, sota_mean + sota_std,
               alpha=0.15, color='black', zorder=1)

    # Y-axis limits
    all_v = [v for v in vals if pd.notna(v)] + [sota_mean]
    if all_v:
        vmin, vmax = min(all_v), max(all_v)
        span = vmax - vmin if vmax > vmin else vmax * 0.1
        ax.set_ylim(max(0, vmin - span * 0.2), vmax + span * 0.2)

    short = [n.replace('pairformer_', 'pf_').replace('pairformer', 'pf_base')
             for n, _, _, _ in MODEL_VARIANTS]
    ax.set_xticks(range(n_bars))
    ax.set_xticklabels(short, rotation=35, ha='right', fontsize=10)
    ax.set_title(f'{task}\n({category})', fontsize=13, fontweight='bold')
    ax.set_ylabel(f'{metric_name} {arrow}', fontsize=11)
    ax.tick_params(axis='y', labelsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, axis='y', ls='--', alpha=0.3)

# Legend
legend_handles = [Patch(facecolor=c, label=f'{n} ({d}d/{dp}L)')
                  for n, d, dp, c in MODEL_VARIANTS]
legend_handles.append(Line2D([0], [0], color='black', linewidth=2.5, linestyle=':', label='TDC SOTA'))
fig.legend(handles=legend_handles, loc='lower center', ncol=len(legend_handles),
           fontsize=10, bbox_to_anchor=(0.5, -0.1), frameon=True)

fig.suptitle('Pairformer Size Ablation (Scratch, No Pre-training)',
             fontsize=15, fontweight='bold')
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()

### 3. Scaling trend: performance vs model size

In [ ]:
# Approximate parameter counts (from config analysis)
PARAM_COUNTS = {
    'pairformer': 5,
    'pairformer_17M': 40,
    'pairformer_52M': 100,
    'pairformer_boltz': 155,
}

fig, axes = plt.subplots(1, len(TASK_ORDER), figsize=(28, 5))

for ax, task in zip(axes, TASK_ORDER):
    col, direction, category, metric_name = TASKS[task]
    arrow = '\u2191' if direction == 'higher' else '\u2193'
    sota_mean, sota_std = TDC_SOTA[task]

    xs, ys, colors = [], [], []
    for name, dim, depth, color in MODEL_VARIANTS:
        v = get_val(scratch, task, col, variant=name)
        if pd.notna(v):
            xs.append(PARAM_COUNTS[name])
            ys.append(v)
            colors.append(color)

    ax.scatter(xs, ys, c=colors, s=120, zorder=5, edgecolors='black', linewidth=0.8)

    # Connect with line
    if len(xs) > 1:
        order = np.argsort(xs)
        ax.plot([xs[i] for i in order], [ys[i] for i in order],
                color='grey', linewidth=1, linestyle='--', alpha=0.5, zorder=1)

    # SOTA line
    ax.axhline(y=sota_mean, color='black', linewidth=2, linestyle=':', alpha=0.7)

    ax.set_xlabel('Params (M)', fontsize=11)
    ax.set_ylabel(f'{metric_name} {arrow}', fontsize=11)
    ax.set_title(f'{task}\n({category})', fontsize=13, fontweight='bold')
    ax.tick_params(labelsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, ls='--', alpha=0.3)

fig.suptitle('Pairformer Scaling: Performance vs Model Size (Scratch)',
             fontsize=15, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

### 4. Effect of pre-training on pairformer_boltz

In [ ]:
# Compare pairformer_boltz: scratch vs FT (rxrx3) vs FT (toymix)
boltz_scratch = scratch[scratch['variant'] == 'pairformer_boltz']

# Separate FT by pretrain dataset
pf_ft_all = pf[(pf['variant'] == 'pairformer_boltz') & pf['is_ft']].copy()
pf_ft_all['pt_type'] = pf_ft_all['pretrain_dataset'].apply(
    lambda s: 'rxrx3' if 'rxrx3' in str(s).lower() else
              ('toymix' if 'toymix' in str(s).lower() else 'other')
)

conditions = [
    ('Scratch', boltz_scratch, '#999999'),
]
for pt_type, color in [('toymix', '#466eff'), ('rxrx3', '#64B478')]:
    sub = pf_ft_all[pf_ft_all['pt_type'] == pt_type]
    if len(sub) > 0:
        conditions.append((f'FT ({pt_type})', sub, color))

print(f'Conditions: {[c[0] for c in conditions]}')

fig, axes = plt.subplots(1, len(TASK_ORDER), figsize=(28, 6))
n_cond = len(conditions)

for ax, task in zip(axes, TASK_ORDER):
    col, direction, category, metric_name = TASKS[task]
    arrow = '\u2191' if direction == 'higher' else '\u2193'
    sota_mean, sota_std = TDC_SOTA[task]

    vals = []
    for label, sub, color in conditions:
        v = pd.to_numeric(sub[sub['task'] == task][col], errors='coerce').dropna()
        vals.append(v.iloc[-1] if len(v) > 0 else np.nan)

    # Find best
    finite = [(i, v) for i, v in enumerate(vals) if pd.notna(v)]
    best_idx = None
    if finite:
        best_idx = (min if direction == 'lower' else max)(finite, key=lambda x: x[1])[0]

    for i, (v, (label, _, color)) in enumerate(zip(vals, conditions)):
        if pd.notna(v):
            ec = 'black' if i == best_idx else color
            lw = 2.5 if i == best_idx else 0.5
            ax.bar(i, v, color=color, edgecolor=ec, linewidth=lw, width=0.6)

    ax.axhline(y=sota_mean, color='black', linewidth=2.5, linestyle=':', zorder=5)
    ax.axhspan(sota_mean - sota_std, sota_mean + sota_std,
               alpha=0.15, color='black', zorder=1)

    all_v = [v for v in vals if pd.notna(v)] + [sota_mean]
    if all_v:
        vmin, vmax = min(all_v), max(all_v)
        span = vmax - vmin if vmax > vmin else vmax * 0.1
        ax.set_ylim(max(0, vmin - span * 0.2), vmax + span * 0.2)

    ax.set_xticks(range(n_cond))
    ax.set_xticklabels([c[0] for c in conditions], rotation=25, ha='right', fontsize=10)
    ax.set_title(f'{task}\n({category})', fontsize=13, fontweight='bold')
    ax.set_ylabel(f'{metric_name} {arrow}', fontsize=11)
    ax.tick_params(axis='y', labelsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, axis='y', ls='--', alpha=0.3)

legend_handles = [Patch(facecolor=c, label=l) for l, _, c in conditions]
legend_handles.append(Line2D([0], [0], color='black', linewidth=2.5, linestyle=':', label='TDC SOTA'))
fig.legend(handles=legend_handles, loc='lower center', ncol=len(legend_handles),
           fontsize=11, bbox_to_anchor=(0.5, -0.08), frameon=True)

fig.suptitle('Pairformer Boltz (384d/48L): Scratch vs Pre-trained',
             fontsize=15, fontweight='bold')
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()

### 5. Training time comparison

In [ ]:
# Load from the benchmark TSV (has timing data)
from pathlib import Path
tsv_path = Path('../results/pairformer_size_benchmark.tsv')
if tsv_path.exists():
    bench = pd.read_csv(tsv_path, sep='\t')
    bench['time_min'] = pd.to_numeric(bench['time_s'], errors='coerce') / 60

    # Pivot: model x task
    time_pivot = bench.pivot_table(index='model', columns='task', values='time_min')

    model_order = ['pairformer', 'pairformer_17M', 'pairformer_52M']
    model_order = [m for m in model_order if m in time_pivot.index]
    time_pivot = time_pivot.loc[model_order]

    fig, ax = plt.subplots(figsize=(10, 5))
    time_pivot.T.plot(kind='bar', ax=ax, width=0.7,
                      color=[c for n, _, _, c in MODEL_VARIANTS if n in model_order])
    ax.set_ylabel('Training Time (minutes)', fontsize=12)
    ax.set_title('Training Time per Task by Pairformer Variant', fontsize=14, fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=10)
    ax.legend(fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, axis='y', ls='--', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print(f'Benchmark TSV not found at {tsv_path}')

### 6. Pairformer vs Pairmixer: speed and quality comparison

Head-to-head on identical setup (384d/48L, 3 epochs, 50 steps, batch 16, ToyMix).

In [ ]:
# Pairformer vs Pairmixer benchmark (same config: 384d/48L, 3 epochs, 50 steps, bs=16)
bench_data = {
    'model':       ['pairformer_boltz', 'pairmixer_boltz'],
    'wall_time_s': [228, 130],
    'peak_mem_MiB':[4416, 4416],
    'train_loss':  [0.76953125, 0.78125],
    'qm9_mae':    [0.76953125, 0.78125],
    'zinc_mae':   [0.796875, 0.79296875],
    'tox21_auroc':[0.30373678, 0.25264552],
}
bench = pd.DataFrame(bench_data).set_index('model')

colors = {'pairformer_boltz': '#9B59B6', 'pairmixer_boltz': '#2ca02c'}
models = bench.index.tolist()

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

# --- Panel 1: Wall time ---
ax = axes[0]
bars = ax.bar(models, bench['wall_time_s'], color=[colors[m] for m in models],
              edgecolor='white', width=0.5)
for bar, val in zip(bars, bench['wall_time_s']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f'{val}s', ha='center', va='bottom', fontsize=12, fontweight='bold')
speedup = bench.loc['pairformer_boltz', 'wall_time_s'] / bench.loc['pairmixer_boltz', 'wall_time_s']
ax.set_title(f'Wall Time (Pairmixer {speedup:.1f}x faster)', fontsize=13, fontweight='bold')
ax.set_ylabel('Seconds', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xticklabels([m.replace('_boltz', '\n(boltz)') for m in models], fontsize=11)

# --- Panel 2: QM9 MAE ---
ax = axes[1]
bars = ax.bar(models, bench['qm9_mae'], color=[colors[m] for m in models],
              edgecolor='white', width=0.5)
for bar, val in zip(bars, bench['qm9_mae']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('QM9 MAE (lower = better)', fontsize=13, fontweight='bold')
ax.set_ylabel('MAE', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xticklabels([m.replace('_boltz', '\n(boltz)') for m in models], fontsize=11)
vmin = min(bench['qm9_mae'])
ax.set_ylim(vmin * 0.95, max(bench['qm9_mae']) * 1.02)

# --- Panel 3: ZINC MAE ---
ax = axes[2]
bars = ax.bar(models, bench['zinc_mae'], color=[colors[m] for m in models],
              edgecolor='white', width=0.5)
for bar, val in zip(bars, bench['zinc_mae']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('ZINC MAE (lower = better)', fontsize=13, fontweight='bold')
ax.set_ylabel('MAE', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xticklabels([m.replace('_boltz', '\n(boltz)') for m in models], fontsize=11)
vmin = min(bench['zinc_mae'])
ax.set_ylim(vmin * 0.995, max(bench['zinc_mae']) * 1.005)

# --- Panel 4: Tox21 AUROC ---
ax = axes[3]
bars = ax.bar(models, bench['tox21_auroc'], color=[colors[m] for m in models],
              edgecolor='white', width=0.5)
for bar, val in zip(bars, bench['tox21_auroc']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Tox21 AUROC (higher = better)', fontsize=13, fontweight='bold')
ax.set_ylabel('AUROC', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xticklabels([m.replace('_boltz', '\n(boltz)') for m in models], fontsize=11)
vmin = min(bench['tox21_auroc'])
ax.set_ylim(vmin * 0.9, max(bench['tox21_auroc']) * 1.05)

fig.suptitle('Pairformer vs Pairmixer (384d/48L, same setup)', fontsize=15, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

# Summary table
print(bench.to_string())
print(f"\nPairmixer is {speedup:.1f}x faster with comparable quality.")
print(f"Same peak memory ({bench.loc['pairmixer_boltz', 'peak_mem_MiB']} MiB).")

### 7. Pairformer vs Pairmixer: downstream ADMET performance

Compares pairformer_boltz and pairmixer_boltz on ADMET downstream tasks.
Both scratch (no pre-training) and finetuned (toymix pre-trained) when available.

In [ ]:
# Reload fresh data (picks up new runs)
df_fresh = pd.read_csv('../results/experiment_results.csv')
df_fresh['timestamp'] = pd.to_datetime(df_fresh['timestamp'], errors='coerce')

# Filter to pairformer and pairmixer boltz variants
mask = (
    ((df_fresh['model'] == 'pyg:pairformer') & (df_fresh['hidden_dim'] == 384) & (df_fresh['gnn_depth'] == 48)) |
    ((df_fresh['model'] == 'pyg:pairmixer') & (df_fresh['hidden_dim'] == 384) & (df_fresh['gnn_depth'] == 48))
)
boltz = df_fresh[mask].copy()
boltz['arch'] = boltz['model'].map({'pyg:pairformer': 'pairformer_boltz', 'pyg:pairmixer': 'pairmixer_boltz'})
boltz['is_ft'] = boltz['is_finetuning'].fillna(False).astype(bool)

# Build condition label
def condition_label(row):
    if not row['is_ft']:
        return f"{row['arch']} (scratch)"
    pt = str(row.get('pretrain_dataset', '')).lower()
    if 'rxrx3' in pt:
        return f"{row['arch']} (FT rxrx3)"
    elif 'toymix' in pt:
        return f"{row['arch']} (FT toymix)"
    return f"{row['arch']} (FT)"

boltz['condition'] = boltz.apply(condition_label, axis=1)

# Dedup: latest per (task, condition)
boltz = boltz.sort_values('timestamp').drop_duplicates(
    subset=['task', 'condition'], keep='last'
).reset_index(drop=True)

# Filter to ADMET tasks only (exclude multitask)
admet_tasks = set(TASKS.keys())
all_admet = set(TDC_SOTA.keys())
boltz_admet = boltz[boltz['task'].isin(all_admet | admet_tasks)]

print(f"Boltz downstream rows: {len(boltz_admet)}")
print(f"Conditions:")
for c, cnt in boltz_admet['condition'].value_counts().sort_index().items():
    print(f"  {c}: {cnt} tasks")

In [ ]:
# Results table: pairformer_boltz vs pairmixer_boltz downstream
DOWNSTREAM_TASKS = TASKS  # reuse the 5 representative tasks

conditions_order = sorted(boltz_admet['condition'].unique())

# Build table
table_rows = []
for task in TASK_ORDER:
    col, direction, category, metric_name = DOWNSTREAM_TASKS[task]
    arrow = '\u2191' if direction == 'higher' else '\u2193'
    sota_mean, sota_std = TDC_SOTA[task]

    row = {
        'Task': task,
        'Category': category,
        'Metric': f'{metric_name} {arrow}',
        'TDC SOTA': f'{sota_mean:.3f} \u00b1 {sota_std:.3f}',
    }
    for cond in conditions_order:
        sub = boltz_admet[(boltz_admet['task'] == task) & (boltz_admet['condition'] == cond)]
        vals = pd.to_numeric(sub[col], errors='coerce').dropna()
        row[cond] = vals.iloc[-1] if len(vals) > 0 else np.nan
    table_rows.append(row)

results_downstream = pd.DataFrame(table_rows).set_index('Task')

# Style: highlight best per row among all conditions
val_cols = [c for c in results_downstream.columns if c not in ('Category', 'Metric', 'TDC SOTA')]

def highlight_best_downstream(row):
    task = row.name
    _, direction, _, _ = DOWNSTREAM_TASKS[task]
    styles = [''] * len(row)
    cols = list(results_downstream.columns)
    vals = {c: row[c] for c in val_cols if pd.notna(row[c])}
    if vals:
        best = min(vals, key=vals.get) if direction == 'lower' else max(vals, key=vals.get)
        j = cols.index(best)
        styles[j] = 'font-weight: bold; color: #1a5fb4'
    return styles

def sota_style_ds(col):
    if col.name == 'TDC SOTA':
        return ['background-color: #f0f0f0; color: #555; font-size: 0.9em'] * len(col)
    return [''] * len(col)

styled_ds = (
    results_downstream.style
    .format(precision=4, na_rep='(pending)', subset=val_cols)
    .apply(highlight_best_downstream, axis=1)
    .apply(sota_style_ds, axis=0)
    .set_caption('Pairformer vs Pairmixer (boltz): Downstream ADMET Performance')
    .set_table_styles([
        {'selector': 'th', 'props': [('font-weight', 'bold')]},
        {'selector': 'td', 'props': [('text-align', 'center')]},
    ])
)
print('Blue = best across all conditions')
display(styled_ds)

In [ ]:
# Per-task bar chart: pairformer_boltz vs pairmixer_boltz
cond_colors = {
    'pairformer_boltz (scratch)': '#D4A5E8',
    'pairformer_boltz (FT toymix)': '#9B59B6',
    'pairformer_boltz (FT rxrx3)': '#7D3C98',
    'pairmixer_boltz (scratch)': '#82E0AA',
    'pairmixer_boltz (FT toymix)': '#2ca02c',
    'pairmixer_boltz (FT rxrx3)': '#1B7A2B',
}

fig, axes = plt.subplots(1, len(TASK_ORDER), figsize=(28, 6))
n_cond = len(conditions_order)

for ax, task in zip(axes, TASK_ORDER):
    col, direction, category, metric_name = DOWNSTREAM_TASKS[task]
    arrow = '\u2191' if direction == 'higher' else '\u2193'
    sota_mean, sota_std = TDC_SOTA[task]

    vals = [results_downstream.loc[task, c] if c in results_downstream.columns else np.nan
            for c in conditions_order]

    # Find best
    finite = [(i, v) for i, v in enumerate(vals) if pd.notna(v)]
    best_idx = None
    if finite:
        best_idx = (min if direction == 'lower' else max)(finite, key=lambda x: x[1])[0]

    for i, (v, cond) in enumerate(zip(vals, conditions_order)):
        color = cond_colors.get(cond, '#999999')
        if pd.notna(v):
            ec = 'black' if i == best_idx else color
            lw = 2.5 if i == best_idx else 0.5
            ax.bar(i, v, color=color, edgecolor=ec, linewidth=lw, width=0.65)

    # SOTA line
    ax.axhline(y=sota_mean, color='black', linewidth=2.5, linestyle=':', zorder=5)
    ax.axhspan(sota_mean - sota_std, sota_mean + sota_std,
               alpha=0.15, color='black', zorder=1)

    # Y-axis limits
    all_v = [v for v in vals if pd.notna(v)] + [sota_mean]
    if all_v:
        vmin, vmax = min(all_v), max(all_v)
        span = vmax - vmin if vmax > vmin else vmax * 0.1
        ax.set_ylim(max(0, vmin - span * 0.2), vmax + span * 0.2)

    # Separator between architectures
    pf_count = sum(1 for c in conditions_order if c.startswith('pairformer'))
    if pf_count > 0 and pf_count < n_cond:
        ax.axvline(x=pf_count - 0.5, color='grey', linewidth=1, linestyle='--', alpha=0.5)

    short = [c.replace('pairformer_boltz', 'PF').replace('pairmixer_boltz', 'PM')
             for c in conditions_order]
    ax.set_xticks(range(n_cond))
    ax.set_xticklabels(short, rotation=40, ha='right', fontsize=8)
    ax.set_title(f'{task}\n({category})', fontsize=13, fontweight='bold')
    ax.set_ylabel(f'{metric_name} {arrow}', fontsize=11)
    ax.tick_params(axis='y', labelsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, axis='y', ls='--', alpha=0.3)

# Legend
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=cond_colors.get(c, '#999'), label=c) for c in conditions_order]
legend_handles.append(Line2D([0], [0], color='black', linewidth=2.5, linestyle=':', label='TDC SOTA'))
fig.legend(handles=legend_handles, loc='lower center', ncol=min(len(legend_handles), 4),
           fontsize=10, bbox_to_anchor=(0.5, -0.12), frameon=True)

fig.suptitle('Pairformer vs Pairmixer (boltz): Downstream ADMET Performance',
             fontsize=15, fontweight='bold')
plt.tight_layout(rect=[0, 0.06, 1, 0.95])
plt.show()

# Win count
print("\nWin count (best per task):")
for cond in conditions_order:
    wins = 0
    for task in TASK_ORDER:
        _, direction, _, _ = DOWNSTREAM_TASKS[task]
        task_vals = {c: results_downstream.loc[task, c] for c in val_cols
                     if pd.notna(results_downstream.loc[task, c])}
        if task_vals:
            best = min(task_vals, key=task_vals.get) if direction == 'lower' else max(task_vals, key=task_vals.get)
            if best == cond:
                wins += 1
    print(f"  {cond}: {wins}/{len(TASK_ORDER)}")